# Character Networks

In this notebook we will construct a specific type of co-occurrence network, a **character network**, for the play *Julius Caesar* by William Shakespeare. In this network, nodes will represent characters in the play and edges will indicate interactions between them. This approach allows us to analyse the structure of relationships in the narrative and identify key characters based on their network position.

Note this notebook requires the *BeautifulSoup* library to be installed (https://pypi.org/project/beautifulsoup4/). You can install this in the terminal or command line by running:

> pip install beautifulsoup4

In [ ]:
from collections import Counter
import networkx as nx
import pandas as pd
# visualisation imports
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

## Data Preprocessing

We are going to parse the HTML file, which contains the entire play. The *BeautifulSoup* library provides tools for extracting structured information from HTML documents.

In [ ]:
from bs4 import BeautifulSoup
# read the HTML file
with open("julius-caesar.html","r") as fin:
    html = fin.read()
    # apply the BeautifulSoup parser to the HTML
    soup = BeautifulSoup(html, "html.parser")

We will separate the play into acts, and the acts into scenes. For each scene, we will extract the sequence of characters who speak during the scene. This hierarchical structure allows us to analyse character interactions at different levels of granularity.

In [ ]:
acts = []
current_act = None
current_scene = None
body_tag = soup.find("body")
# process all tags inside the page body
for tag in body_tag.find_all():
    # start of an act?
    if tag.name == "h2":
        current_act = []
        acts.append( current_act )
    # start of a scene?
    elif tag.name == "h3":  
        current_scene = []
        current_act.append( current_scene )
    # start of character dialogue?
    elif tag.name == "p" and "class" in tag.attrs and "speaker" in tag.attrs["class"]:
        # tidy up the charactername
        char_name = tag.text.strip().title() 
        current_scene.append( char_name )

Check how many acts and scenes we have found:

In [ ]:
for i, act in enumerate(acts):
    print(f"Act {i+1} has {len(act)} scenes")

## Network Construction

We are going to create a separate network for each act in the play. This approach allows us to observe how character relationships evolve throughout the narrative.

We will use a "co-linear" approach, where we create an edge between each successive pair of characters who speak in a scene. This method captures the immediate conversational interactions between characters.

In [ ]:
# process each act
act_networks = []
for act in acts:
    # process each scene in the act, counting the character pairs
    counts = Counter()
    for scene in act:
        # get each successive pair of characters
        num_speakers = len(scene)
        for pos in range(1,num_speakers):
            # get the next sequence of 2 characters
            char_name1 = scene[pos-1]
            char_name2 = scene[pos]
            # skip if they're the same character
            if char_name1 == char_name2:
                continue
            # convert to a set, because the order doesn't matter
            pair = frozenset([char_name1, char_name2])
            counts[pair] += 1
    # now actually create the network from the counts
    g = nx.Graph()
    for pair in counts:
        g.add_edge(*pair, weight=counts[pair])
    act_networks.append(g)

## Network Characterisation

We are now going to perform some basic characterisation of the different network acts, and then compile it all into a Pandas DataFrame for easy comparison. This systematic approach allows us to quantify structural differences between acts.

In [ ]:
rows = []
for i, g in enumerate(act_networks):
    row = { "act" : (i+1) }
    row["nodes"] = g.number_of_nodes()
    row["edges"] = g.number_of_edges()
    row["density"] = nx.density(g)
    row["components"] = nx.number_connected_components(g)
    rows.append(row)

In [ ]:
# create the actual Data Frame
df = pd.DataFrame(rows).set_index("act")
df

Let's draw one of the networks as an example:

In [ ]:
plt.figure(figsize=(11,6))
plt.margins(0.1, 0.1)
nx.draw(act_networks[1], 
        with_labels=True, 
        node_size=1000, 
        font_size=12, 
        node_color="#a0ddba")
plt.show()

Next, we look at the most central characters in each act, based on degree centrality. This measure identifies characters who have the most direct connections to other characters:

In [ ]:
for i, g in enumerate(act_networks):
    # calculate weighted degrees
    degrees = dict(g.degree())
    # convert to a Pandas series and sort it
    sdeg = pd.Series(degrees, name="degree")
    sdeg = sdeg.sort_values(ascending=False)
    # display the top results
    print(f"Act {i+1}")
    display(pd.DataFrame(sdeg.head(3)))

We will repeat the process, but using weighted degree centrality. This measure considers not only the number of connections but also the strength of those connections:

In [ ]:
for i, g in enumerate(act_networks):
    # calculate weighted degrees
    wdegrees = dict(g.degree(weight="weight"))
    # convert to a Pandas series and sort it
    swdeg = pd.Series(wdegrees, name="wdegree")
    swdeg = swdeg.sort_values(ascending=False)
    # display the top results
    print(f"Act {i+1}")
    display(pd.DataFrame(swdeg.head(3)))

Let's merge the separate networks for the different acts into a single network. To do this we need to sum the weights from the different networks, creating a comprehensive view of character interactions throughout the entire play:

In [ ]:
# merge the weights  
merged_counts = Counter()
for g in act_networks:
    for e in g.edges(data=True):
        pair = frozenset([e[0],e[1]])
        merged_counts[pair] += e[2]["weight"]
# now actually create the network from the counts
g_overall = nx.Graph()
for pair in merged_counts:
    g_overall.add_edge(*pair, weight=merged_counts[pair])

In [ ]:
g_overall.number_of_nodes(), g_overall.number_of_edges()

Look at the degree centrality in the overall network:

In [ ]:
degrees = dict(g_overall.degree())
# convert to a Pandas series and sort it
sdeg = pd.Series(degrees, name="degree")
sdeg = sdeg.sort_values(ascending=False)
# display the top 10
pd.DataFrame(sdeg.head(10))

Repeat the process, but now look at weighted degree centrality for the overall network:

In [ ]:
wdegrees = dict(g_overall.degree(weight="weight"))
# convert to a Pandas series and sort it
swdeg = pd.Series(wdegrees, name="wdegree")
swdeg = swdeg.sort_values(ascending=False)
# display the top 10
pd.DataFrame(swdeg.head(10))

Based on both centrality measures, it appears that Caesar is actually not the most central character in the play *Julius Caesar*. The character of Brutus, who betrayed Caesar, is more central to the narrative. This finding shows how network analysis can reveal insights that might not be immediately obvious from a casual reading of the text.

We can also look at the weighted degree distribution for the overall network:

In [ ]:
ax = swdeg.plot.hist(figsize=(10, 5), fontsize=12, legend=None, color="darkred", bins=25, zorder=3)
ax.set_ylabel("Number of Nodes", fontsize=12)
ax.set_xlabel("Degree", fontsize=12)
ax.set_xlim(0)
ax.yaxis.grid()
plt.show()

From the plot it is clear that we have two characters who are central to the narrative (i.e. Brutus and Cassius), a large cast of lesser characters, and a collection of peripheral characters who only appear a few times in the play. This hierarchical structure is typical of dramatic works.

## Ego Networks

Next, we will create and visualise ego networks for some of the central characters in the overall network. Ego networks provide focused views of individual characters' immediate social environments in the play.

Firstly, we will define a utility function to extract and draw an ego network:

In [ ]:
def display_ego(g, ego_node):
    # build the ego network
    eg = nx.ego_graph(g, ego_node)
    # create the figure
    plt.figure(figsize=(9,7))
    plt.margins(0.1, 0.1)
    title = f"Ego network for {ego_node} ({eg.number_of_nodes()} nodes, {eg.number_of_edges()} edges)"
    plt.title(title, fontsize=12)
    # lay out all nodes
    pos = nx.spring_layout(g)
    # draw the full network
    nx.draw_networkx(eg, pos, 
                     with_labels=True, 
                     font_size=12, 
                     node_size=900, 
                     node_color="lightblue")
    # draw the ego in red, with larger node size
    nx.draw_networkx_nodes(eg, pos, 
                           nodelist=[ego_node], 
                           node_size=2500, 
                           node_color="red")
    plt.axis("off")
    plt.show()

Now use this function to create the ego networks for Brutus, Julius Caesar and Antony. These visualisations highlight the different patterns of interaction for each character:

In [ ]:
display_ego(g_overall, "Brutus")

In [ ]:
display_ego(g_overall, "Caesar")

In [ ]:
display_ego(g_overall, "Antony")